In [31]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
df = pd.read_csv("WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [33]:
df.shape

(1470, 35)

In [34]:
df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

In [35]:
df.duplicated().sum()

np.int64(0)

In [36]:
# Attrition sütununu binary-yə çevirmək (Yes=1, No=0)
df['Attrition_Binary'] = df['Attrition'].map({'Yes': 1, 'No': 0})

In [37]:
# 2. Ümumi attrition rate-in hesablanması
total_employees = len(df)
total_attritions = df['Attrition_Binary'].sum()
overall_attrition_rate = (total_attritions / total_employees) * 100

In [38]:
print(f"Ümumi işçi sayı: {total_employees}")
print(f"Şirkəti tərk edənlər (Attrition): {total_attritions}")
print(f"Ümumi Attrition Rate: {overall_attrition_rate:.2f}%")

Ümumi işçi sayı: 1470
Şirkəti tərk edənlər (Attrition): 237
Ümumi Attrition Rate: 16.12%


In [39]:
kpi_data = []
for status, group in df.groupby('Attrition'):
    kpi_data.append({
        'Status': status,
        'Total Employees': len(group),
        'Total Attritions': group['Attrition_Binary'].sum() if status == 'Yes' else 0,
        'Share of Total (%)': round((len(group) / len(df)) * 100, 2),
        'Avg Monthly Income': round(group['MonthlyIncome'].mean(), 2),
        'Avg Years At Company': round(group['YearsAtCompany'].mean(), 2),
        'Avg Job Satisfaction': round(group['JobSatisfaction'].mean(), 2)
    })

kpi_df = pd.DataFrame(kpi_data)
display(kpi_df.style.background_gradient(subset=['Avg Monthly Income', 'Avg Job Satisfaction'], cmap='Blues'))

,Status,Total Employees,Total Attritions,Share of Total (%),Avg Monthly Income,Avg Years At Company,Avg Job Satisfaction
0,No,1233,0,83.880000,6832.740000,7.370000,2.780000
1,Yes,237,237,16.120000,4787.090000,5.130000,2.470000


In [40]:
# Tenure qruplarının yaradılması (0–2 yrs, 3–5 yrs, 6–10 yrs, 11+ yrs)
bins = [-1, 2, 5, 10, df['YearsAtCompany'].max()]
labels = ['0–2 yrs', '3–5 yrs', '6–10 yrs', '11+ yrs']
df['Tenure_Group'] = pd.cut(df['YearsAtCompany'], bins=bins, labels=labels)

# Tenure qrupları üzrə attrition rate hesablanması
tenure_attrition = df.groupby('Tenure_Group', observed=False)['Attrition_Binary'].agg(
    Total_Employees='count',
    Attritions='sum',
    Attrition_Rate=lambda x: round((x.sum() / len(x)) * 100, 2)
).reset_index()

display(tenure_attrition)

,Tenure_Group,Total_Employees,Attritions,Attrition_Rate
0,0–2 yrs,342,102,29.82
1,3–5 yrs,434,60,13.82
2,6–10 yrs,448,55,12.28
3,11+ yrs,246,20,8.13


In [41]:
# Income qruplarının yaradılması (Q1–Q4)
df['Income_Quartile'] = pd.qcut(df['MonthlyIncome'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Income kvartilləri üzrə attrition rate hesablanması
income_attrition = df.groupby('Income_Quartile', observed=False)['Attrition_Binary'].agg(
    Total_Employees='count',
    Attritions='sum',
    Attrition_Rate=lambda x: round((x.sum() / len(x)) * 100, 2)
).reset_index()

display(income_attrition)

,Income_Quartile,Total_Employees,Attritions,Attrition_Rate
0,Q1,369,108,29.27
1,Q2,366,52,14.21
2,Q3,367,39,10.63
3,Q4,368,38,10.33


In [42]:
# Bonus - Cost of Attrition hesablanması (Replacement cost = 6 ay maaş × churned employee sayı)
total_churned = df['Attrition_Binary'].sum()
avg_monthly_income_churned = df[df['Attrition_Binary'] == 1]['MonthlyIncome'].mean()
total_attrition_cost = total_churned * (avg_monthly_income_churned * 6)

print(f"Cəmi ayrılan işçi sayı: {total_churned}")
print(f"İllik ümumi attrition xərci: ${total_attrition_cost:,.2f}")

Cəmi ayrılan işçi sayı: 237
İllik ümumi attrition xərci: $6,807,246.00
